# CS383: Data Science and Machine Learning
## Lecture 10 — Tree-Based Models: Decision Trees, Random Forests, and Gradient Boosting

*Dr. Thitima Srivatanakul*

### Guiding question
**You've built two very different classifiers -- one that fits a straight-line formula, one that looks
at nearby neighbors. What if a classifier could just ask a series of yes/no questions instead?**

### Learning objectives
By the end of this lecture, you should be able to:

- explain how a decision tree makes a prediction by asking a sequence of yes/no questions about features
- explain why a single, unconstrained decision tree tends to overfit, and how limiting its depth trades
  fit for generalization
- explain how a random forest combines many trees (bagging) to make that overfitting less severe
- explain, at a high level, how gradient boosting builds trees one at a time, each one correcting the
  previous ensemble's mistakes
- fit and use `DecisionTreeClassifier`, `RandomForestClassifier`, and XGBoost's `XGBClassifier` with
  scikit-learn's familiar `.fit()` / `.predict()` interface
- extract and interpret feature importances from a tree-based model, including a caution about what
  "importance" does and doesn't mean
- compare a tree-based model's test performance against a baseline and against Lecture 8's logistic
  regression, using the majority-baseline habit from Lectures 8 and 9

---

### Where this fits
Lecture 8 gave you your first two classifiers -- logistic regression's straight-line boundary and
k-NN's local, neighbor-based one. Lecture 9 gave you the tools to judge any classifier fairly. Today
adds a third, very different way to draw a decision boundary: a tree of yes/no questions. Part 1 builds
a single decision tree on Lecture 8's toy reinspection dataset and visualizes its boundary next to the
straight line and the curvy neighbor-based one you've already seen. Part 2 scales that idea up to a
random forest -- many trees, voting. Part 3 adds gradient boosting (via XGBoost) and tests all three
tree-based approaches, plus feature importance, on the same real restaurant inspections data from
Lectures 6-9.

---

### Before we open the notebook: an unplugged warm-up

No coding for this part. You'll play "20 Questions" against a small set of restaurants to build a
decision tree by hand, see why a tree that asks too many questions memorizes its training set instead of
learning from it, and get an intuitive, no-math preview of why averaging many trees (or building them one
correction at a time) tends to work better than trusting a single one.

**[Open the "Twenty Questions" Activity](https://thitimas.github.io/cs383-fa26-materials-public/lect10/trees_unplugged_activity.html)**

Takes about 15-20 minutes. Come back here once you've been through all the rounds.

---

## Part 1 — Decision Trees

Same toy dataset as Lecture 8, Part 2: 40 restaurants that failed an inspection and were scheduled for a
reinspection, described by two features -- violations racked up, and days until the reinspection. You've
already seen logistic regression draw a single straight line through this data, and k-NN draw a boundary
that hugs individual points. This time: what if the classifier just asked yes/no questions about the
features, one at a time?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

VIOLATIONS = np.array([2, 0, 9, 8, 2, 9, 2, 8, 5, 3, 5, 0, 5, 2, 5, 3, 2, 5, 2, 5,
                       7, 0, 5, 6, 0, 6, 6, 5, 8, 10, 4, 10, 9, 7, 0, 10, 4, 0, 8, 6], dtype=float)
DAYS_UNTIL_REINSPECTION = np.array([258, 228, 273, 47, 323, 302, 322, 24, 256, 37, 24, 218, 24, 8, 341,
                                     114, 184, 229, 99, 310, 6, 351, 296, 374, 377, 137, 365, 104, 70,
                                     259, 155, 324, 225, 321, 167, 128, 298, 344, 347, 340], dtype=float)
PASSES_REINSPECTION = np.array([1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0,
                                 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0])

X_toy = np.__________([VIOLATIONS, DAYS_UNTIL_REINSPECTION])
print(f"{len(PASSES_REINSPECTION)} restaurants, {PASSES_REINSPECTION.sum()} passed reinspection")


def plot_decision_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 20, X[:, 1].max() + 20
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))

    grid_preds = model.predict(np.column_stack([xx.ravel(), yy.ravel()]))
    grid_preds = grid_preds.reshape(xx.shape)

    plt.contourf(xx, yy, grid_preds, levels=[-0.5, 0.5, 1.5], colors=["#F7D9D6", "#D6E4F0"], alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=70, edgecolor="white", linewidth=1)
    plt.xlabel("Violations")
    plt.ylabel("Days until reinspection")
    plt.title(title)
    plt.show()

### How a decision tree decides

A decision tree makes a prediction by asking one yes/no question about a feature, following the branch
that matches the answer, and repeating -- until it lands on a leaf that just says "pass" or "fail." Fitting
a tree means choosing which question to ask at each step: the one that splits the training data into the
two most "pure" groups possible (as close to all-pass or all-fail as it can manage). No formula, no
probabilities required -- just a flowchart, built automatically from the data.

In [ ]:
shallow_tree = DecisionTreeClassifier(max_depth=2, random_state=383)
shallow_tree.__________(X_toy, PASSES_REINSPECTION)

print(export_text(shallow_tree, feature_names=["violations", "days_until_reinspection"]))

Read that top to bottom like a flowchart: "is `violations` at most some cutoff? If yes, go left and ask
a second question; if no, go right and ask a different one." Follow any restaurant's two numbers down the
tree and you land on a leaf that predicts pass or fail, with a class count showing how many training
restaurants landed there too. Every branch is a single yes/no question about a single feature -- that's
the entire model.

In [ ]:
plot_decision_boundary(shallow_tree, X_toy, PASSES_REINSPECTION, "Decision Tree Boundary (max_depth=2)")

Compare this to Lecture 8's two boundaries on this exact same data: logistic regression drew one
straight line; k-NN drew a boundary that curves smoothly around individual points. This one is made of
rectangles with straight, axis-aligned edges -- because every single split tests just one feature against
one cutoff, the boundary can only ever bend at a right angle, never diagonally or smoothly. That blocky
look isn't a limitation of this particular tree; it's what *every* decision tree boundary looks like.

### What happens with no depth limit?

`max_depth=2` only allowed the tree to ask two questions before committing to an answer. What if it could
ask as many questions as it wanted, splitting all the way down until every leaf is pure?

In [ ]:
deep_tree = DecisionTreeClassifier(random_state=383)  # no max_depth -- grows until every leaf is pure
deep_tree.fit(X_toy, PASSES_REINSPECTION)

print(f"Tree depth: {deep_tree.get_depth()}")
print(f"Training accuracy: {deep_tree.__________(X_toy, PASSES_REINSPECTION):.3f}")

plot_decision_boundary(deep_tree, X_toy, PASSES_REINSPECTION, "Decision Tree Boundary (unrestricted depth)")

100% training accuracy, and a boundary carved into tiny little boxes around individual points -- the same
warning sign as k-NN with $k=1$ back in Lecture 8. An unrestricted tree keeps asking questions until every
training point is isolated into its own pure leaf, which means it's memorized the training set rather than
learned the underlying pattern. `max_depth` (and a handful of similar settings you'll meet again in
Lecture 11) is how you rein that in -- deliberately stopping the tree from asking every possible question,
trading perfect training accuracy for a boundary that actually generalizes.

---

## Part 2 — Random Forests: Many Trees, One Vote

A single deep tree overfits. A single shallow tree might underfit -- two questions isn't much to work
with. Instead of picking one tree and hoping its depth is exactly right, a **random forest** grows many
trees and lets them vote:

1. build hundreds of trees, each on a random *bootstrap sample* of the training data (drawn with
   replacement, so no two trees see quite the same rows)
2. at each split, each tree is only allowed to consider a random subset of the features
3. for a new point, every tree casts a vote for a class; the forest predicts whichever class gets the
   most votes

Each individual tree in the forest is still free to grow deep and a little overfit -- but because every
tree overfits to a *different* random slice of the data, their mistakes tend to disagree with each other
and cancel out when you average the votes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=__________, random_state=383)
forest.fit(X_toy, PASSES_REINSPECTION)

plot_decision_boundary(forest, X_toy, PASSES_REINSPECTION, "Random Forest Boundary (100 trees)")

Still boxy -- it's built from the same kind of yes/no splits -- but noticeably smoother than the single
unrestricted tree's boundary. That's the same story as k=1 vs. k=15 in Lecture 8, arrived at a different
way: instead of asking one model to look at more neighbors, a random forest trains *many* separately
overfit models and averages their votes. One tree's noise is another tree's non-issue; a hundred trees'
worth of noise mostly averages out, leaving the real pattern behind. `n_estimators` -- how many trees to
grow -- is the main dial: more trees generally means a smoother, more stable boundary, at the cost of
more computation.

---

## Part 3 — Gradient Boosting and Feature Importance on Real Data

Random forests build many trees *independently* and average them (a strategy called **bagging**).
**Gradient boosting** builds trees *sequentially* instead: fit one small tree, see where it's still
wrong, fit the next tree specifically to correct those remaining mistakes, add it to the ensemble, and
repeat -- often hundreds of times. Each new tree is trained to chip away at whatever error the ensemble
built so far still has, rather than each tree working from scratch on its own random slice of the data.

**XGBoost** is a fast, heavily optimized implementation of gradient boosting, and one of the most widely
used tools for exactly this kind of tabular (rows-and-columns) prediction problem in real-world data
science. It follows the same scikit-learn-style `.fit()` / `.predict()` interface you already know.

Let's test all three tree-based approaches -- and feature importance -- on the real restaurant
inspections data from Lectures 6-9.

### Setup — NYC restaurant inspections

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

try:
    raw_path = os.path.expanduser("~/shared/restaurant_inspections_snapshot.csv")
    inspections_df = pd.read_csv(raw_path)
    inspections_df["score"] = pd.to_numeric(inspections_df["score"], errors="coerce")
    inspections_df = inspections_df.dropna(subset=["score", "boro", "cuisine_description"]).reset_index(drop=True)
    inspections_df["is_critical"] = (inspections_df["critical_flag"] == "Critical").astype(int)

    top_cuisines = inspections_df["cuisine_description"].value_counts().nlargest(10).index
    inspections_df["cuisine_top"] = np.where(
        inspections_df["cuisine_description"].isin(top_cuisines), inspections_df["cuisine_description"], "Other"
    )
    live = True

except Exception:
    rng = np.random.default_rng(383)
    n = 4000
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza", "Japanese"]

    inspections_df = pd.DataFrame({
        "boro": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_top": rng.choice(cuisines_clean, size=n),
        "score": rng.integers(0, 71, size=n),
    })
    inspections_df["is_critical"] = rng.integers(0, 2, size=n)
    live = False

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(inspections_df):,} violation records")
print(f"is_critical balance: {inspections_df['is_critical'].mean():.3f}")

In [ ]:
X = inspections_df[["score", "boro", "cuisine_top"]]
y = inspections_df["is_critical"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=383, stratify=y)

majority_class = y_train.__________()[0]
majority_baseline = (y_test == majority_class).mean()
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")

One setup step you *won't* see below: scaling `score`. A tree's split -- "is `score` at most 12.5?" --
only cares about the order of values, not their units or range, so scaling changes nothing about where a
tree decides to split. That's a real, practical difference from k-NN back in Lecture 8, which needed
`score` on the same footing as everything else because it measures raw distance. The only preprocessing
these models still need is one-hot encoding `boro` and `cuisine_top`, since trees (like every model
here) still need numbers in, not text.

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["boro", "cuisine_top"]),
], remainder="passthrough")

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.__________(X_test)

tree_model = DecisionTreeClassifier(max_depth=4, random_state=383)
tree_model.fit(X_train_ready, y_train)

tree_pred = tree_model.predict(X_test_ready)
print(f"Decision tree accuracy: {accuracy_score(y_test, tree_pred):.3f}  (baseline: {majority_baseline:.3f})")

A single depth-4 tree, same ballpark as logistic regression's roughly 59% from Lecture 8 on these same
three features -- a modest step up from the baseline, not a dramatic one. That's a useful data point
already: whatever's limiting these models isn't the *shape* of the decision boundary (straight line vs.
tree of rectangles), since both land in about the same place.

In [ ]:
# max_depth=4 here too -- bagging makes overfitting less severe, but it doesn't erase it, so
# an unrestricted forest can still do worse than a depth-limited one on a weak-signal problem like this.
forest_model = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=383)
forest_model.__________(X_train_ready, y_train)

forest_pred = forest_model.predict(X_test_ready)
print(f"Random forest accuracy: {accuracy_score(y_test, forest_pred):.3f}  (baseline: {majority_baseline:.3f})")

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=383, eval_metric="logloss")
xgb_model.fit(X_train_ready, y_train)

xgb_pred = xgb_model.__________(X_test_ready)
print(f"XGBoost accuracy:       {accuracy_score(y_test, xgb_pred):.3f}  (baseline: {majority_baseline:.3f})")

Decision tree, random forest, and XGBoost all land in roughly the same narrow band above the baseline,
not far from logistic regression's number from Lecture 8 either. That's a genuinely useful, if slightly
deflating, finding: swapping in a more expressive model -- one that can carve curved, complicated
boundaries instead of a single straight line -- doesn't manufacture signal that isn't in `score`, `boro`,
and `cuisine_top` to begin with. More powerful models can squeeze out real, if modest, gains when there
are non-linear patterns or feature interactions a straight line can't capture; they can't invent
information the features never had. That's the same honest-signal lesson from Lecture 8's real-data
section, holding up under a much stronger set of models.

### Which features is the model actually using?

Every tree-based model above chose, at each split, which feature best separated the classes -- which
means it's kept a running tally of how useful each feature turned out to be. That tally is exposed as
`.feature_importances_`.

In [ ]:
feature_names = preprocessor.get_feature_names_out()
importances = pd.Series(xgb_model.__________, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(7, 5))
importances.head(10).plot(kind="barh", color="#2B6CB0")
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("XGBoost Feature Importances (top 10)")
plt.tight_layout()
plt.show()

importances.head(10)

`score` should sit at or near the top -- consistent with it being the only feature with a clear,
direct relationship to `is_critical` back in Lecture 8. A few individual boroughs or cuisines may show up
with small non-zero importance, but nowhere close to `score`.

Two cautions before you trust an importance ranking too far. First, it's specific to *this* model and
*this* data -- a different algorithm, or a different train/test split, can reorder the smaller
importances even when the big picture stays the same. Second, importance measures how useful a feature
was for making *splits*, which isn't the same thing as real-world causal importance: two correlated
features can end up splitting credit between them in a way that understates both, and a feature with many
distinct values (like `cuisine_top` after one-hot encoding) sometimes gets more chances to look useful
just by having more columns to split on. Treat feature importance as a genuinely useful clue about what
the model is doing, not as proof of what actually matters in the real world.

---

**Exercises for this lecture** (Lab, Exit Ticket, Optional Challenge) live in a separate notebook:
`lect10_tree_models_exercise.ipynb`.

---

## Cheat Sheet

| Task | Code |
|---|---|
| Fit a decision tree | `DecisionTreeClassifier(max_depth=d).fit(X_train, y_train)` |
| Read a tree as text | `export_text(model, feature_names=[...])` |
| Fit a random forest | `RandomForestClassifier(n_estimators=n).fit(X_train, y_train)` |
| Fit an XGBoost classifier | `XGBClassifier(n_estimators=n, max_depth=d).fit(X_train, y_train)` |
| Feature importances | `model.feature_importances_` |
| Get one-hot feature names back | `preprocessor.get_feature_names_out()` |
| Accuracy | `accuracy_score(y_test, y_pred)` |
| Majority-class baseline | `(y_test == y_train.mode()[0]).mean()` |

---

## Key Terms

- **Decision tree**: a model that predicts by asking a sequence of yes/no questions about features,
  ending in a leaf that predicts a class; its boundary is always made of axis-aligned rectangles.
- **Overfitting (in a tree)**: an unrestricted tree keeps splitting until every training point is
  isolated, memorizing the training set instead of learning the general pattern; `max_depth` is one way
  to limit this.
- **Bagging**: training many models independently, each on a random bootstrap sample of the data, and
  averaging (or voting on) their predictions to smooth out any one model's overfitting.
- **Random forest**: a bagged ensemble of decision trees, each trained on a random sample of rows and a
  random subset of features at each split; predicts by majority vote.
- **Gradient boosting**: an ensemble built sequentially -- each new tree is trained to correct the
  errors the ensemble made so far, rather than being trained independently.
- **XGBoost**: a fast, widely-used implementation of gradient boosting for tabular data, following
  scikit-learn's `.fit()` / `.predict()` interface.
- **Feature importance**: a score, per feature, of how useful that feature was for the splits a
  tree-based model actually made; a useful clue about the model, not a guarantee of real-world causal
  importance.